# CLM-0.4-mini M1 Infrastructure

This notebook validates the M1 data/model/variant/checkpoint plumbing only. It must not run development seed `90401` or formal seeds `90411/90412/90413`. The only allowed decision is `SMOKE_ONLY`.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/ArcheLabs/mini-cells.git'
REPO_REF = 'main'
KAGGLE_WORKING = Path('/kaggle/working')

cwd = Path.cwd().resolve()
if (cwd / 'pyproject.toml').is_file():
    ROOT = cwd
else:
    ROOT = KAGGLE_WORKING / 'mini-cells'
    if not ROOT.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', REPO_REF,
            REPO_URL, str(ROOT),
        ], check=True)
    elif not (ROOT / 'pyproject.toml').is_file():
        raise RuntimeError(f'{ROOT} exists but is not a MiniCells checkout')
    else:
        subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', REPO_REF, '--depth', '1'], check=True)
        subprocess.run(['git', '-C', str(ROOT), 'checkout', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', REPO_REF], check=True)

os.chdir(ROOT)
assert (ROOT / 'pyproject.toml').is_file()
assert (ROOT / 'scripts' / 'research' / 'run.py').is_file()
print('Repository root:', ROOT)
subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev,lm]', '-q'], check=True, cwd=ROOT)


In [ ]:
OUT = ROOT / 'results' / 'clm-0.4-mini-m1-infrastructure'
subprocess.run([
    sys.executable, str(ROOT / 'scripts' / 'research' / 'run.py'),
    'clm-0.4-mini-m1', '--smoke', '--device', 'cpu', '--out', str(OUT),
], check=True, cwd=ROOT)
subprocess.run([
    sys.executable, str(ROOT / 'scripts' / 'research' / 'report.py'),
    'clm-0.4-mini-m1', '--results', str(OUT),
], check=True, cwd=ROOT)


In [ ]:
import json
summary_path = OUT / 'summary.json'
assert summary_path.is_file(), summary_path
summary = json.loads(summary_path.read_text())
assert summary['status'] == 'SMOKE_ONLY'
assert summary['development_seed_observed'] is False
assert summary['formal_seeds_observed'] is False
summary['formal_model_parameter_count'], summary['transaction_projection_ids']


## Formal data preparation

The seed-independent M1 data identity now uses the pinned TinyStories revision `f54c09fd23315a6f9c86f9dc80f725de7d8f9c64`, frozen in `calibration-assets.json`. The repository does not commit the 30M-token corpus. Build it from the repository root with:

```bash
cd /kaggle/working/mini-cells
python scripts/research/prepare_clm_0_4_mini_data.py \
  --dataset-revision f54c09fd23315a6f9c86f9dc80f725de7d8f9c64 \
  --routing-salt clm-0.4-mini-v1 \
  --out /kaggle/working/clm-0.4-mini-data
```

This step is seed-independent. The resulting hashes must match `research/validations/clm-0.4-mini-language-validation/calibration-assets.json` before `90401` is opened.